# Proposed validation — review before it runs

**What this measures:** The target metric is the worst-case absolute logit gap between the real natten-free port (all four released checkpoints loaded through `timm.create_model`/`checkpoint_filter_fn`) and a freshly-cloned LMMMEng/OverLoCK reference executing natten's `na2d_av` on one identical seeded fp32 224x224 batch — that gap is exactly the claimed `F.unfold` ≡ `na2d_av` equivalence, with guardrails pinning post-filter missing/unexpected key counts to exactly 0 (both replaced ops are parameter-free, so any nonzero key is a porting bug). Tension to note: neither the reference repo's code nor a natten version is visible in this context, so the reference is cloned at branch `main`, `natten` comes unpinned from PyPI, and the reference's per-variant factory entry points are resolved by signature introspection that crashes loudly if absent; the ImageNet-1k shard is best-effort per the user's "if available" and never gates.

**Target metric:** `max_abs_logit_diff_vs_natten_ref`

Remyx wrote this test for the change in this PR. **Nothing here has been executed** — there are no outputs, and no result is being claimed.

Edit it if the measurement is wrong, then mention `@remyx validate` again and it will run what you committed. If anything is missing at run time — an import, a dependency, a device — the run reports it and repairs what it can rather than failing silently.

This notebook is the only copy of the test: `.remyx/validation.yaml` points here, and the run executes these cells top to bottom.

In [ ]:
# papermill parameters — Remyx injects variant / ref / seed here
variant = ""
ref = ""
seed = 0

## Execution context

The cells below are the test Remyx drafted, split into cells; it was written as a script, so `__file__` is set to `eval/eval_overlock_weight_parity.py` (no such file is committed). This cell gives it what the command line would: its own path in `__file__`, an empty argument list so `argparse` sees no stray flags, and the papermill parameters as `REMYX_VARIANT` / `REMYX_REF` / `REMYX_SEED` for anything that wants them.

In [ ]:
import os, sys
ROOT = os.getcwd()  # the notebook runs with the repository root as its working directory
__file__ = os.path.join(ROOT, "eval/eval_overlock_weight_parity.py")
sys.argv = [__file__]
for _k in ("variant", "ref", "seed"):
    _v = globals().get(_k)
    if _v not in (None, ""):
        os.environ["REMYX_" + _k.upper()] = str(_v)
print("[remyx] cwd", ROOT, "| script", __file__)

In [ ]:
"""OverLoCK weight-parity eval (single arm; question.kind=capability, baseline=none).

User guidance (binding GO gate): for each overlock_{xt,t,s,b}, the released in1k_224 checkpoint
must load into the natten-free timm port with zero missing/unexpected keys after the real
`checkpoint_filter_fn`, and port logits must match a freshly-built natten reference
(LMMMEng/OverLoCK @ main, natten installed) within atol=1e-3 on a fixed seeded fp32 batch.
Optional best-effort: overlock_xt top-1 on a streaming ImageNet-1k val shard (-1.0 if unavailable).
Mirrors tests/test_models.py idioms (create_model, resolve_data_config/create_transform).

Baseline safety: every overlock module this diff ADDS (`timm.models.overlock` with its
`checkpoint_filter_fn`, and the `models.overlock` import) is guarded defensively
(ImportError/AttributeError only). On a pre-PR checkout they fail cleanly and main() emits
degraded sentinels that fail every gate instead of crashing; nothing else is swallowed.
"""
import argparse
import importlib.metadata as md
import inspect
import json
import os
import subprocess
import sys
import tempfile
import traceback
from types import ModuleType

In [ ]:
REPO_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
sys.path.insert(0, REPO_ROOT)

import torch

# The modules/symbols the diff ADDS, imported defensively per the repo's own guard idiom.
# Baseline arm: import fails cleanly -> main() emits degraded metrics. PR head: any real
# failure below this line raises with a traceback (no blanket handlers on the measurement).
try:
    from timm.models.overlock import checkpoint_filter_fn
except (ImportError, AttributeError):
    checkpoint_filter_fn = None

In [ ]:
_P = argparse.ArgumentParser()
_P.add_argument('--variant')  # accepted + ignored: variants are fixed by the spec
_P.add_argument('--ref')
_P.add_argument('--seed', type=int, default=0)
ARGS = _P.parse_args()

SMOKE = os.environ.get('REMYX_SMOKE') == '1'
VARIANTS = ['xt'] if SMOKE else ['xt', 't', 's', 'b']
BATCH, SHARD_N = (1, 4) if SMOKE else (4, 2048)
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
if torch.cuda.is_available():
    # fixed kernels / no autotune: same inputs -> same logits run-to-run on the GPU
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
# Count natten.functional.na2d_av calls; patched BEFORE the reference import can bind it, so a
# counter that stays at 0 during the reference forward proves the parity test would be vacuous.
import natten.functional as _naf
_na_calls = [0]
_real_na2d_av = _naf.na2d_av
def _counting_na2d_av(*a, **k):
    _na_calls[0] += 1
    return _real_na2d_av(*a, **k)
_naf.na2d_av = _counting_na2d_av

In [ ]:
# Fetch + import the REAL reference implementation (branch main; HEAD sha printed as diagnostic).
REF_ROOT = os.path.join(tempfile.gettempdir(), 'overlock_ref')
if not os.path.isdir(REF_ROOT):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/LMMMEng/OverLoCK', REF_ROOT], check=True)
try:
    import einops  # noqa: F401 -- reference models/contmix.py uses einops; tiny pure-python dep
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'einops'], check=True,
                   stdout=subprocess.DEVNULL)
if 'models' not in sys.modules:  # reach reference models/*.py without executing an unknown __init__
    _pkg = ModuleType('models')
    _pkg.__path__ = [os.path.join(REF_ROOT, 'models')]
    sys.modules['models'] = _pkg
try:
    # guarded exactly like the timm import above: this overlock module is added by the PR, so on a
    # pre-PR baseline checkout it may not exist and the run must degrade to sentinels, not crash.
    # ImportError/AttributeError only -- nothing else is swallowed anywhere in this script.
    import models.overlock as ref_mod
except (ImportError, AttributeError):
    ref_mod = None
_ref_head = subprocess.run(['git', '-C', REF_ROOT, 'rev-parse', 'HEAD'], check=True,
                           capture_output=True, text=True).stdout.strip()
print(f'[ref] LMMMEng/OverLoCK HEAD={_ref_head} natten=={md.version("natten")} device={DEV}', file=sys.stderr)

In [ ]:
def _unwrap(raw):
    sd = raw.get('state_dict', raw)
    return {(k[7:] if k.startswith('module.') else k): v for k, v in sd.items()}

def build_reference(v):
    cls = getattr(ref_mod, 'OverLoCK', None)
    if cls is None:
        raise RuntimeError(f'no OverLoCK class in {ref_mod.__file__}: '
                           f'{[n for n in vars(ref_mod) if n[:1].isupper()]}')
    factory = getattr(cls, f'overlock_{v}', None) or getattr(ref_mod, f'overlock_{v}', None)
    if factory is None or not callable(factory):
        raise RuntimeError(f'reference exposes no overlock_{v} factory: '
                           f'{[n for n in dir(cls) if n.startswith("overlock")]}')
    params = inspect.signature(factory).parameters
    kw = {'num_classes': 1000} if ('num_classes' in params or any(
        p.kind is inspect.Parameter.VAR_KEYWORD for p in params.values())) else {}
    return factory(**kw).eval().to(DEV)

In [ ]:
def load_reference(model, raw):
    inc = model.load_state_dict(_unwrap(raw), strict=False)
    bad = [k for k in list(inc.missing_keys) + list(inc.unexpected_keys) if not k.startswith('aux_head')]
    if bad:
        raise RuntimeError(f'reference build mismatches checkpoint outside aux_head.*: {bad[:6]}')

def _emit(**kw):
    print(json.dumps({
        'max_abs_logit_diff_vs_natten_ref': kw.get('max_abs_logit_diff_vs_natten_ref', 1.0),
        'missing_keys_after_filter': kw.get('missing_keys_after_filter', 1),
        'unexpected_keys_after_filter': kw.get('unexpected_keys_after_filter', 1),
        'argmax_agreement_frac': kw.get('argmax_agreement_frac', 0.0),
        'aux_head_keys_dropped': kw.get('aux_head_keys_dropped', 0),
        'overlock_xt_top1': kw.get('overlock_xt_top1', -1.0),
        'imagenet_shard_n': kw.get('imagenet_shard_n', 0),
    }))

In [ ]:
def main():
    if checkpoint_filter_fn is None or ref_mod is None:
        # Baseline (pre-PR) checkout: the PR-added overlock module is absent, the defensive
        # imports failed cleanly. Degraded sentinels fail every gate; no crash, JSON intact.
        print('[baseline] overlock module absent (pre-PR checkout): emitting degraded '
              'sentinels (max_diff=1.0, missing=unexpected=1)', file=sys.stderr)
        _emit()
        return

    import timm  # the changed module under test is exercised via create_model + filter above
    inputs = torch.randn(BATCH, 3, 224, 224, generator=torch.Generator().manual_seed(ARGS.seed)).to(DEV)
    max_diff, agree, seen, miss, unex, aux = 0.0, 0, 0, 0, 0, 0
    for v in VARIANTS:
        model = timm.create_model(f'overlock_{v}', pretrained=True).eval().to(DEV)  # released .pth via pretrained_cfg
        raw = torch.hub.load_state_dict_from_url(model.default_cfg['url'], map_location='cpu')  # torch.hub cache hit
        filtered = checkpoint_filter_fn(raw, model)
        aux += sum(1 for k in _unwrap(raw) if k.startswith('aux_head') and k not in filtered)
        inc = model.load_state_dict(filtered, strict=False)
        miss += len(inc.missing_keys)
        unex += len(inc.unexpected_keys)
        with torch.inference_mode():
            port_logits = model(inputs)
        del model
        torch.cuda.empty_cache()

        ref = build_reference(v)
        load_reference(ref, raw)
        before = _na_calls[0]
        with torch.inference_mode():
            ref_logits = ref(inputs)
        if isinstance(ref_logits, (tuple, list)):
            ref_logits = ref_logits[0]
        calls = _na_calls[0] - before
        if ref_logits.shape != port_logits.shape or calls == 0:
            raise RuntimeError(f'overlock_{v}: ref logits {tuple(ref_logits.shape)} vs port '
                               f'{tuple(port_logits.shape)}, na2d_av calls={calls} '
                               f'(0 => reference did not run natten; parity would be vacuous)')
        d = (port_logits.float() - ref_logits.float()).abs().max().item()
        max_diff = max(max_diff, d)
        agree += int((port_logits.argmax(1) == ref_logits.argmax(1)).sum())
        seen += BATCH
        print(f'[overlock_{v}] max|dlogit|={d:.3e} missing={len(inc.missing_keys)} '
              f'unexpected={len(inc.unexpected_keys)} na2d_av_calls={calls}', file=sys.stderr)
        del ref
        torch.cuda.empty_cache()

    top1, shard_n = -1.0, 0
    try:  # optional shard per user guidance: 'if an ImageNet-1k val shard is available'
        import datasets as hfds
        import timm.data
        ds = hfds.load_dataset('ILSVRC/imagenet-1k', split='validation', streaming=True)
        model = timm.create_model('overlock_xt', pretrained=True).eval().to(DEV)
        pp = timm.data.create_transform(**timm.data.resolve_data_config(model=model))
        correct, xs, ys = 0, [], []
        def _flush():
            nonlocal correct, shard_n
            with torch.inference_mode():
                pred = model(torch.stack(xs).to(DEV)).argmax(1).cpu()
            correct += int((pred == torch.tensor(ys)).sum())
            shard_n += len(ys)
        for i, ex in enumerate(ds):
            if i >= SHARD_N:
                break
            xs.append(pp(ex['image']))
            ys.append(ex['label'])
            if len(xs) == 32:
                _flush()
                xs, ys = [], []
        if xs:
            _flush()
        if shard_n:
            top1 = 100.0 * correct / shard_n
    except Exception:
        traceback.print_exc()  # shard unavailable -> top1=-1.0, shard_n=0; the gates above are unaffected

    _emit(
        max_abs_logit_diff_vs_natten_ref=max_diff,  # target: < 1e-3 (user-fixed atol; expected fp32 reassociation drift ~1e-6..1e-5)
        missing_keys_after_filter=miss,             # guardrail: == 0 (na2d_av/iGEMM carry zero parameters)
        unexpected_keys_after_filter=unex,          # guardrail: == 0 after the documented aux_head drop
        argmax_agreement_frac=(agree / seen) if seen else 0.0,
        aux_head_keys_dropped=aux,
        overlock_xt_top1=top1,
        imagenet_shard_n=shard_n,
    )

if __name__ == '__main__':
    main()

## The criteria this is judged against

From `.remyx/validation.yaml` — thresholds live here, not in the test, so a failing measurement reports rather than crashes.

```yaml
question:
  kind: capability
  ask: "Do all four released overlock_{xt,t,s,b}_in1k_224.pth checkpoints load into the natten-free timm port with zero missing/unexpected keys after checkpoint_filter_fn, and do the port's logits match a freshly-built natten reference within atol=1e-3?"
baseline: none
loop:
  max_iterations: 8
  fix_code: true
benchmarks:
  - name: overlock-weight-parity
    suite: "eval/eval_overlock_weight_parity.py"
    packages: [natten, datasets]
    scorer: max_abs_logit_diff_vs_natten_ref
    metrics:
      - name: max_abs_logit_diff_vs_natten_ref
        direction: min
        threshold: 0.001
        role: target
        bar: goal
        reads_as: "worst-case |port logit minus natten-reference logit| across all four variants on the fixed fp32 batch; must sit under 1e-3"
        # atol=1e-3 fixed by user_guidance; expected fp32 reassociation drift over <=361-term neighborhood sums is ~1e-6..1e-5, orders below the band
      - name: missing_keys_after_filter
        direction: min
        threshold: 0.5
        role: guardrail
        bar: goal
        reads_as: "checkpoint parameters the port lacks after checkpoint_filter_fn (integer count summed over all four variants); any nonzero value is a porting bug"
        # the value is a non-negative integer, so the ONLY integer satisfying <= 0.5 is 0: the bound
        # is exactly 'zero missing keys across all four released checkpoints', which is the correct
        # behaviour fixed by user_guidance (na2d_av/iGEMM are parameter-free, so the port key set
        # must equal the checkpoint modulo the aux_head drop). It fails on 1+ and on the pre-PR
        # degraded sentinel (1) -- a real bound, not a vacuous '>= 0'.
      - name: unexpected_keys_after_filter
        direction: min
        threshold: 0.5
        role: guardrail
        bar: goal
        reads_as: "checkpoint keys the port does not consume after filtering (aux_head already dropped); integer count over all four variants; any nonzero value is a bug"
        # same anchoring: integer count, the only passing value is 0 (exact key parity after the
        # documented aux_head drop); a single stray key (1) or the degraded sentinel (1) fails it
      - name: argmax_agreement_frac
        direction: max
        threshold: 1.0
        role: diagnostic
        reads_as: "fraction of batch positions where port and natten reference pick the same class; 1.0 expected, a flip only means near-ties"
      - name: aux_head_keys_dropped
        direction: max
        threshold: 1
        role: diagnostic
        reads_as: "training-only aux_head.* keys dropped by checkpoint_filter_fn on the real checkpoints, proving the filter ran on them rather than a toy dict"
        # every released in1k checkpoint carries the deep-supervision aux head, so correct behaviour
        # drops at least one aux_head key; 0 -- also the degraded-sentinel value -- means the filter
        # never ran on the real checkpoints and fails this bound
      - name: overlock_xt_top1
        direction: max
        threshold: 81.9
        role: diagnostic
        reads_as: "overlock_xt top-1 in percent on the ImageNet-1k val shard with timm default_cfg preprocessing; -1.0 means no shard was obtainable"
        # floor 81.9 = assumed paper XT row 82.2 minus 0.3pp (paper table row NOT visible in context; diagnostic only, never gates)
      - name: imagenet_shard_n
        direction: max
        threshold: 1
        role: diagnostic
        reads_as: "number of ImageNet-1k val images actually scored; >= 1 makes overlock_xt_top1 a real measurement, 0 means the shard was unavailable"
        # full mode asks for SHARD_N=2048 images; >= 1 distinguishes 'a real accuracy measurement'
        # from the top1=-1.0 sentinel. 0 is an accepted best-effort outcome (user guidance: 'if
        # available') and never gates: this metric is a diagnostic, not a target or guardrail.
    policy:
      guardrail_veto: true
compute:
  tier: gpu
  # one arm: 4 checkpoint downloads (~0.85 GB from GitHub release v1) + 4 port builds + 4 reference builds
  # + B=4 fp32 224x224 forwards through both models ≈ 15 min; pip natten ≤10 min (wheel) up to ~40 min (source build);
  # optional ~2048-image streaming ImageNet-1k val shard ≈ 20 min -> 7200 s budget
  timeout_s: 7200
held_constant:
  - "identical fixed input batch (seeded torch.randn, 224x224, fp32, cudnn.deterministic=True, benchmark=False) through port and natten reference, so preprocessing is pinned by construction"
  - "same released checkpoint file per variant (GitHub release v1 URL wired into pretrained_cfg) feeds both the port and the reference"
  - "both models eval() + inference_mode, fp32, same CUDA device, same strict-load policy after dropping training-only aux_head.* keys"
  - "optional shard scored with timm default_cfg preprocessing (crop_pct, interpolation, mean/std)"
avoid:
  - "reference LMMMEng/OverLoCK cloned at branch main since no commit is visible in context; its HEAD sha is printed as a diagnostic, not pinned"
  - "natten installed unpinned from PyPI (no version visible in context); a torch/cuda wheel mismatch may force a long source build"
  - "ImageNet-1k shard streamed at the dataset repo's default revision (no hash visible in context; inventing one would 404) — diagnostic only, never gates"
  - "no wall-clock timing anywhere — parity, key counts and accuracy only"
  - "ImageNet shard acquisition is best-effort per the user's 'if available'; failure yields imagenet_shard_n=0 and top1=-1.0 and must not fail the gate metrics (traceback still printed to stderr)"
  - "on a pre-PR baseline checkout both PR-added overlock modules are absent: timm.models.overlock and models.overlock each fail their defensive import cleanly (ImportError/AttributeError only) and the run prints degraded sentinels (max_diff=1.0, missing=unexpected=1, agreement=0.0) that fail every gate rather than crashing; no other handler swallows measurement errors"
  - "guardrails encode exact-zero key parity as <= 0.5 on the integer counts (only 0 passes; 1+ and the degraded sentinel fail), fixed by user_guidance; the pre-filter aux_head drop is expected and reported via aux_head_keys_dropped (>= 1 proves the filter ran on the real checkpoints)"
provenance:
  max_abs_logit_diff_vs_natten_ref: "user_guidance"
  missing_keys_after_filter: "user_guidance"
  unexpected_keys_after_filter: "user_guidance"
  overlock_xt_top1: "user_guidance"
  argmax_agreement_frac: "inferred"
  aux_head_keys_dropped: "inferred"
  imagenet_shard_n: "inferred"
  held_constant: "user_guidance"
  suite: "synthesized"
  compute: "user_guidance"
report:
  headline: "All four released checkpoints load clean into the natten-free port and its logits sit inside the 1e-3 band of the natten reference"
  findings:
    - "Worst |port minus natten| logit gap across overlock_{xt,t,s,b} on the fixed batch: {max_abs_logit_diff_vs_natten_ref} (gate 1e-3)"
    - "After checkpoint_filter_fn: {missing_keys_after_filter} missing and {unexpected_keys_after_filter} unexpected keys (gate: both exactly 0, encoded as <= 0.5 on the integer count)"
    - "Argmax agreement with the natten reference on the fixed batch: {argmax_agreement_frac:%}"
    - "Training-only aux_head.* keys dropped by the filter on the real checkpoints: {aux_head_keys_dropped}"
    - "overlock_xt top-1 on {imagenet_shard_n} ImageNet-1k val images: {overlock_xt_top1} percent (paper XT row assumed 82.2, band 0.3pp, -1.0 means no shard)"
  establishes:
    - "F.unfold + replicate-pad value aggregation computes the same function as natten's na2d_av on the released weights, within 1e-3"
    - "full state-dict key parity modulo the documented aux_head drop, exercised on all four released checkpoints rather than a toy dict"
  does_not_establish:
    - "sub-1e-3 numerical drift, training-time behaviour, deploy/reparam folding, or downstream COCO/ADE20K results"
  not_measured:
    - "throughput, latency, memory, TorchScript export (excluded by the PR), features_only pyramid values"
  caveat: "Paper top-1 was trained WITH natten and deep supervision, the XT row (82.2) is assumed rather than read from the paper table, and a shard under ~6.5k images cannot statistically resolve the 0.3pp band"
  next: "If any key mismatch or logit divergence appears, fix the port before upstreaming; otherwise run the full 50k ImageNet val pass for the accuracy row"
```